# ViAmpleHate on Kaggle: XLM-RoBERTa + ViCLSR

Notebook này chạy 2 model theo 2 link được gửi:

- `FacebookAI/xlm-roberta-base`: baseline XLM-RoBERTa cho text classification.
- `huynhtin/ViCLSR`: checkpoint ViCLSR từ paper `arXiv:2603.21084`, dùng đúng MLP projection head rồi fine-tune classifier cho hate speech.

Khuyến nghị Kaggle: bật `Internet`, chọn GPU `T4`. ViCLSR là XLM-RoBERTa-Large nên nặng hơn, batch nhỏ là bình thường.

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "kaggle" / "run_two_models.py").exists():
    if not (ROOT / "ViAmpleHate").exists():
        subprocess.run(["git", "clone", "https://github.com/MinhTuan2405/ViAmpleHate.git"], check=True)
    os.chdir(ROOT / "ViAmpleHate")

ROOT = Path.cwd()
assert (ROOT / "kaggle" / "run_two_models.py").exists(), "Không tìm thấy kaggle/run_two_models.py"
print("Repo path:", ROOT)

In [ ]:
!pip install -q -r kaggle/requirements.txt

## Cấu hình

Đổi `DATASET` thành `"vihsd"` hoặc `"vozhsd"`.

- `vihsd`: dùng split train/validation/test có sẵn từ `sonlam1102/vihsd`, map `CLEAN/OFFENSIVE -> NON-HATE`, `HATE -> HATE`.
- `vozhsd`: dataset `tarudesu/VOZ-HSD` chỉ có split train, script tự chia stratified `75/12.5/12.5`.

In [ ]:
DATASET = "vihsd"   # "vihsd" hoặc "vozhsd"
EPOCHS = 3
MAX_LEN = 256
OUTPUT_DIR = "/kaggle/working/viamplehate_runs"

# Chỉ dùng khi DATASET="vozhsd".
# proposed: mirror notebook Proposed VOZ-HSD: sample 40k, HATE 10%, split 75/12.5/12.5.
# baseline: mirror baseline VOZ-HSD: giữ tỉ lệ tự nhiên, split 80/10/10.
VOZ_SPLIT_POLICY = "proposed"  # "proposed" hoặc "baseline"
VOZ_SAMPLE_SIZE = 40_000
VOZ_HATE_RATIO = 0.10


## Smoke Test

Chạy 2 cell này trước để kiểm tra path, dataset, model loading, forward/backward, save metrics. XLM-RoBERTa smoke test nhanh; ViCLSR smoke test vẫn phải tải checkpoint lớn lần đầu.

In [ ]:
!python kaggle/run_two_models.py \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs 1 \
  --max-len 64 \
  --batch-size 2 \
  --eval-batch-size 4 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

In [ ]:
!python kaggle/run_two_models.py \
  --dataset {DATASET} \
  --model viclsr \
  --epochs 1 \
  --max-len 64 \
  --batch-size 1 \
  --eval-batch-size 2 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

## 1. Chạy XLM-RoBERTa

Model này nhẹ hơn ViCLSR vì dùng `xlm-roberta-base`. Nếu GPU yếu, giảm `--batch-size 4`.

In [ ]:
!python kaggle/run_two_models.py \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs {EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 8 \
  --eval-batch-size 16 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}

## 2. Chạy ViCLSR

ViCLSR là XLM-RoBERTa-Large và checkpoint khoảng vài GB. Cell này dùng batch nhỏ. Nếu CUDA OOM, đổi `--max-len 128` hoặc giữ `--batch-size 1`.

In [ ]:
!python kaggle/run_two_models.py \
  --dataset {DATASET} \
  --model viclsr \
  --epochs {EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 1 \
  --eval-batch-size 2 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}

## 3. Xem metrics

Mỗi model lưu `best_model.pt`, `metrics.json`, và tokenizer vào `/kaggle/working`.

In [ ]:
import json
from pathlib import Path

base = Path(OUTPUT_DIR) / DATASET
for model_name in ["xlm-roberta", "viclsr"]:
    path = base / model_name / "metrics.json"
    print("\n===", model_name, "===")
    if not path.exists():
        print("Chưa có metrics:", path)
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print({k: metrics[k] for k in ["accuracy", "macro_f1", "hate_f1"]})
    print(metrics["report"])

## Ghi chú báo cáo

Khi viết report, ưu tiên so sánh `macro_f1` và `hate_f1`, vì class `HATE` ít hơn nhiều so với `NON-HATE`.